# 01 - Contextmethoden en Prompting (Nederlandse Editie)

Deze notebook demonstreert de evolutie van het verstrekken van context aan AI-modellen, met praktijkvoorbeelden gericht op Nederland.

## 🇳🇱 Over Deze Nederlandse Versie
Deze notebook bevat voorbeelden specifiek voor de Nederlandse context:
- Nederlandse steden (Amsterdam, Rotterdam, Utrecht, Den Haag, Eindhoven)
- Nederlandse bedrijven en organisaties (NS, Schiphol, KPN)
- Lokale regelgeving (AVG/GDPR, NEN normen)
- Euro-valuta en Europese tijdzone

## 🧠 Waarom Context Belangrijk Is

Zonder context is AI beperkt tot zijn trainingsdata. Overweeg deze scenario's:

| Vraag | Zonder Context | Met Context |
|-------|---------------|-------------|
| "Hoe is het weer?" | Kan niet antwoorden (weet locatie niet) | "In Amsterdam is het 18°C en bewolkt" |
| "Welke laptop moet ik kopen?" | Generiek advies | Gepersonaliseerde aanbeveling op basis van behoeften |
| "Hoe optimaliseer ik mijn database?" | Algemene tips | Specifieke aanbevelingen gebaseerd op jouw stack |

## 🎯 Leerdoelen
- De evolutie van contextvoorziening begrijpen
- Leren wanneer verschillende methoden te gebruiken
- Praktische ervaring opdoen met Nederlandse use cases
- Voorbereiden op function calling en MCP

In [ ]:
import os
from openai import AzureOpenAI
from dotenv import load_dotenv
load_dotenv()

# Azure OpenAI Configuratie
client = AzureOpenAI(
    azure_endpoint=os.getenv("AZURE_OPENAI_ENDPOINT"),
    api_key=os.getenv("AZURE_OPENAI_API_KEY"),
    api_version="2024-05-01-preview"
)

deployment = os.getenv("AZURE_OPENAI_GPT4O_DEPLOYMENT_NAME")

print("✅ Azure OpenAI client geïnitialiseerd")
print(f"🤖 Model deployment: {deployment}")

## 🚫 Methode 1: Basis Chat (Geen Context)

Laten we eerst zien wat er gebeurt wanneer AI geen context heeft. Let op de generieke, niet-specifieke antwoorden.

In [ ]:
def basic_chat_example():
    """Demonstreer de beperkingen van chat zonder context"""
    messages = [
        {
            "role": "user",
            "content": "Hoe is het weer?"
        }
    ]
    
    response = client.chat.completions.create(
        model=deployment,
        messages=messages,
        max_tokens=300,
        temperature=0.7
    )
    
    print("🤖 AI Response (Zonder Context):")
    print("=" * 50)
    print(response.choices[0].message.content)
    print("=" * 50)
    print("\n💡 Let op: AI kan de vraag niet beantwoorden zonder locatiecontext!")
    
    return response

basic_response = basic_chat_example()

## 📋 Methode 2: System Prompt (Statische Context)

System prompts voegen statische context toe die beschikbaar blijft gedurende het hele gesprek. Perfect voor domeinspecifieke assistenten.

### Voorbeeld: Nederlandse Azure Architect

In [ ]:
def system_prompt_example_nl():
    """Demonstreer statische context via system prompts met Nederlandse context"""
    messages = [
        {
            "role": "system",
            "content": """Je bent een Microsoft Azure Solutions Architect met diepgaande expertise in cloud-architectuurpatronen. Je hebt toegang tot de volgende context over de huidige infrastructuur van onze Nederlandse organisatie:

- Omgeving: Productie-workloads in West Europe (Amsterdam) regio
- Huidige Services: 30+ App Services, 5 AKS clusters, 20+ Azure SQL databases
- Maandelijkse Uitgaven: €40.000
- Compliance Vereisten: AVG/GDPR, SOC 2, NEN 7510 (voor zorgdata)
- Ontwikkelteams: 4 teams die .NET Core, Python en Node.js gebruiken
- Locatie: Hoofdkantoor in Amsterdam, bijkantoren in Rotterdam en Utrecht
- Pijnpunten: Hoge database kosten, complexe networking setup, handmatige deployments

Geef architectuuraanbevelingen op basis van deze context. Houd altijd rekening met kostenoptimalisatie, security best practices en onze compliance-vereisten. Antwoord in het Nederlands."""
        },
        {
            "role": "user",
            "content": "We ervaren prestatieproblemen met onze SQL databases tijdens piekuren. Wat zou je aanbevelen?"
        }
    ]
    
    response = client.chat.completions.create(
        model=deployment,
        messages=messages,
        max_tokens=1000,
        temperature=0.7
    )
    
    print("🤖 AI Response (Met System Context):")
    print("=" * 50)
    print(response.choices[0].message.content)
    print("=" * 50)
    print("\n💡 Let op: AI geeft specifieke aanbevelingen op basis van de bedrijfscontext!")
    
    return response

system_response = system_prompt_example_nl()

## 💬 Methode 3: Dynamische Context via Multi-Turn Gesprek

Deze aanpak bouwt progressief context op via gesprek, wat flexibele en verkennende interacties mogelijk maakt.

### Voorbeeld: Nederlandse E-commerce Platform

In [ ]:
def multi_turn_example_nl():
    """Demonstreer dynamische context opbouw via gesprek met Nederlandse context"""
    
    # Gespreksbeurten die progressief context opbouwen
    conversation = [
        {
            "role": "user",
            "content": "Ik werk aan een webapplicatie voor mijn Nederlandse bedrijf."
        },
        {
            "role": "assistant", 
            "content": "Leuk om te horen! Om de beste begeleiding te geven, kun je me meer vertellen over:\n- Welke technologiestack gebruik je?\n- Wat is het hoofddoel van de applicatie?\n- Zijn er specifieke uitdagingen waar je tegenaan loopt?"
        },
        {
            "role": "user",
            "content": "Het is een e-commerce platform gebouwd met Next.js en Node.js. We gebruiken PostgreSQL voor de database en deployen op Azure in de West Europe regio. We hebben ongeveer 15.000 producten en verwachten 8.000 dagelijks actieve gebruikers, voornamelijk uit Nederland en België."
        },
        {
            "role": "assistant",
            "content": "Een e-commerce platform met Next.js, Node.js en PostgreSQL op Azure West Europe is een solide architectuur voor de Nederlandse markt. Met 15.000 producten en 8.000 DAU wil je focussen op prestaties en schaalbaarheid. Welk specifiek aspect kan ik je mee helpen?\n- Prestatie-optimalisatie\n- Database-ontwerp\n- Caching-strategieën\n- Azure deployment configuratie\n- Zoekfunctionaliteit\n- iDEAL/betalingsintegratie\n- AVG-compliance"
        },
        {
            "role": "user",
            "content": "We hebben problemen met trage productzoekfunctie. Gebruikers klagen dat het 3-5 seconden duurt om resultaten te krijgen. We gebruiken momenteel LIKE-queries in PostgreSQL. We moeten ook Nederlandse en Belgische klanten bedienen met ondersteuning voor beide talen."
        }
    ]
    
    response = client.chat.completions.create(
        model=deployment,
        messages=conversation,
        max_tokens=1000,
        temperature=0.7
    )
    
    print("🤖 AI Response (Dynamische Context Opgebouwd via Gesprek):")
    print("=" * 50)
    print(response.choices[0].message.content)
    print("=" * 50)
    print("\n💡 Let op: AI geeft specifieke oplossingen gebaseerd op opgebouwde context!")
    
    return response, conversation

multi_turn_response, conversation_history = multi_turn_example_nl()

## 📊 Vergelijk Contextmethoden

Laten we de verschillen tussen deze benaderingen analyseren:

In [ ]:
def analyze_responses():
    """Vergelijk de verschillende contextmethoden"""
    
    print("📊 CONTEXTMETHODE VERGELIJKING")
    print("=" * 60)
    
    print("\n🚫 BASIS CHAT (Geen Context):")
    print(f"   📝 Tokens gebruikt: {basic_response.usage.total_tokens}")
    print(f"   🎯 Specificiteit: Laag - Generieke responses")
    print(f"   ⚡ Setup tijd: Direct")
    print(f"   🔄 Aanpasbaarheid: Geen")
    
    print("\n📋 SYSTEM PROMPT (Statische Context):")
    print(f"   📝 Tokens gebruikt: {system_response.usage.total_tokens}")
    print(f"   🎯 Specificiteit: Hoog - Domeinspecifiek")
    print(f"   ⚡ Setup tijd: Eenmalige configuratie")
    print(f"   🔄 Aanpasbaarheid: Vast tot prompt verandert")
    
    print("\n💬 MULTI-TURN (Dynamische Context):")
    print(f"   📝 Tokens gebruikt: {multi_turn_response.usage.total_tokens}")
    print(f"   🎯 Specificiteit: Zeer Hoog - Op maat gemaakt")
    print(f"   ⚡ Setup tijd: Bouwt op tijdens gesprek")
    print(f"   🔄 Aanpasbaarheid: Zeer flexibel")
    
    print("\n📈 TOKEN GEBRUIK ANALYSE:")
    total_conversation_tokens = sum(len(msg["content"].split()) * 1.3 for msg in conversation_history)
    print(f"   🚫 Basis: {basic_response.usage.total_tokens} tokens")
    print(f"   📋 System: {system_response.usage.total_tokens} tokens") 
    print(f"   💬 Multi-turn: {multi_turn_response.usage.total_tokens} tokens")
    print(f"   📊 Gespreksgeschiedenis: ~{int(total_conversation_tokens)} tokens")

analyze_responses()

## 🇳🇱 Nederlandse Use Case: NS Reisassistent

Laten we een praktisch voorbeeld bouwen: een NS reisassistent met system prompt context.

In [ ]:
def ns_reisassistent_example():
    """Nederlandse Spoorwegen reisassistent voorbeeld"""
    messages = [
        {
            "role": "system",
            "content": """Je bent een behulpzame NS (Nederlandse Spoorwegen) reisassistent. Je hebt toegang tot de volgende informatie:

**Populaire Routes en Reistijden:**
- Amsterdam Centraal - Rotterdam Centraal: 40 minuten (Intercity Direct), €17,20
- Amsterdam Centraal - Utrecht Centraal: 27 minuten, €8,50
- Amsterdam Centraal - Den Haag Centraal: 50 minuten, €12,80
- Amsterdam Centraal - Eindhoven Centraal: 1 uur 20 minuten, €22,10
- Schiphol Airport - Amsterdam Centraal: 15 minuten, €5,40

**Abonnementen:**
- NS Flex: Betaal per reis, geen vaste kosten
- Dal Voordeel: €5,60/maand - 40% korting buiten de spits (09:00-16:00 en na 18:30)
- Altijd Voordeel: €27/maand - 40% korting altijd
- Weekend Vrij: €35/maand - Onbeperkt reizen in weekenden

**Actuele Informatie:**
- Werkzaamheden Utrecht-Amersfoort dit weekend
- Spitsuren: 07:00-09:00 en 16:30-18:30
- Fiets meenemen buiten spits of met Fiets mee-dagkaart (€7,50)

Geef behulpzaam en vriendelijk advies aan reizigers. Antwoord altijd in het Nederlands."""
        },
        {
            "role": "user",
            "content": "Ik moet morgen van Amsterdam naar Rotterdam voor een vergadering om 09:00. Welke trein moet ik nemen en hoeveel kost het?"
        }
    ]
    
    response = client.chat.completions.create(
        model=deployment,
        messages=messages,
        max_tokens=800,
        temperature=0.7
    )
    
    print("🚂 NS Reisassistent Response:")
    print("=" * 50)
    print(response.choices[0].message.content)
    print("=" * 50)
    
    return response

ns_response = ns_reisassistent_example()

## 🎯 Wanneer Elke Methode Te Gebruiken

### 🚫 **Basis Chat** (Geen Context)
**Het beste voor:**
- Snelle, algemene vragen
- Verkennende gesprekken
- Wanneer context niet nodig is

**Beperkingen:**
- Geen domeinexpertise
- Geen toegang tot real-time data
- Generieke responses

### 📋 **System Prompts** (Statische Context)
**Het beste voor:**
- Domeinspecifieke assistenten
- Consistente gedragsvereisten
- Goed gedefinieerde scenario's
- Productcatalogi, bedrijfsbeleid

**Beperkingen:**
- Alleen statische informatie
- Geen real-time updates
- Beperkte flexibiliteit

### 💬 **Multi-Turn Gesprek** (Dynamische Context)
**Het beste voor:**
- Complexe probleemoplossing
- Requirements verzameling
- Adaptieve assistentie
- Gebruikersvoorkeuren leren

**Beperkingen:**
- Hoger tokengebruik
- Context window limieten
- Vereist meerdere API-aanroepen

## 🚀 Evolutie naar Function Calling

Hoewel deze methoden context bieden, kunnen ze nog steeds niet toegang krijgen tot:
- ⏰ Real-time data (huidige tijd, weer)
- 🌐 Live API's (NS vertrektijden, vliegtuigstatus)
- 💾 Externe databases
- 🔧 Systeemoperaties

**Daar wordt function calling essentieel!**

## 🎓 Wat Je Hebt Geleerd

✅ **Context Fundamenten**: Begrijpen waarom AI context nodig heeft  
✅ **Statische Context**: Hoe system prompts domeinkennis verstrekken  
✅ **Dynamische Context**: Begrip opbouwen via gesprek  
✅ **Trade-offs**: Tokengebruik vs. specificiteit vs. flexibiliteit  
✅ **Use Cases**: Wanneer elke methode toe te passen  
✅ **Nederlandse Context**: Praktische voorbeelden voor Nederland  

## 🚀 Volgende Stappen

Nu je de contextvoorzieningsmethoden begrijpt, laten we verkennen hoe function calling real-time mogelijkheden inschakelt:

- **02_function_calling_legacy_format.ipynb** - Leer het verouderde formaat (educatief)
- **03_function_calling_modern_tools.ipynb** - Beheers huidige best practices  
- **04_parallel_multiple_tool_execution.ipynb** - Geavanceerde parallelle uitvoering
- **05_assistants_api_with_vector_stores.ipynb** - Document-gebaseerde intelligentie

**Klaar om je AI real-time superkrachten te geven?** Laten we doorgaan! 🎯